<a href="https://colab.research.google.com/github/maobiobi/YouTube_Podcast_RAG_3MTT_Capstone/blob/main/YouTube_Podcast_RAG_3MTT_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙 YouTube Podcast RAG  
### 3MTT Data Engineering Capstone Project

This notebook ingests transcripts from up to **5 YouTube podcast episodes**, transforms them into chunks, stores semantic embeddings in a **Chroma vector database**, retrieves relevant chunks for a user query, and generates grounded answers using a **Flan-T5 model**, with **source citation**.

### Pipeline:
**Ingestion → Chunking → Embedding → Vector Storage → Retrieval → Generation**


In [ ]:
!pip uninstall youtube-transcript-api -y
!pip install youtube-transcript-api==0.6.1
!pip install -q --upgrade langchain chromadb sentence-transformers transformers ipywidgets langchain_text_splitters langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2

In [ ]:
import youtube_transcript_api

In [ ]:
!pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 0.6.1
Summary: This is an python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: https://github.com/jdepoix/youtube-transcript-api
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: UNKNOWN
Location: /usr/local/lib/python3.12/dist-packages
Requires: requests
Required-by: 


In [ ]:
import ipywidgets as widgets
widgets.IntSlider()
from IPython.display import display, clear_output, Markdown

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import re

In [ ]:
display(Markdown("# 🎙 YouTube Podcast RAG"))
display(Markdown("### 3MTT Data Engineering Capstone Project"))
display(Markdown("Paste up to 5 YouTube podcast URLs, process transcripts, and ask questions using Retrieval-Augmented Generation (RAG)."))

# 🎙 YouTube Podcast RAG

### 3MTT Data Engineering Capstone Project

Paste up to 5 YouTube podcast URLs, process transcripts, and ask questions using Retrieval-Augmented Generation (RAG).

In [ ]:
from IPython.display import display
import ipywidgets as widgets

box = widgets.Text(value="test")
display(box)

Text(value='test')

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

url_inputs = []

for i in range(5):
    url = widgets.Text(
        placeholder=f"Paste YouTube URL {i+1}",
        description=f"URL {i+1}:",
        layout=widgets.Layout(width='90%')
    )
    url_inputs.append(url)

   # display(url)

    # Display everything together
display(widgets.VBox(url_inputs))

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
def extract_video_id(url):
    if "youtu.be" in url:
        return url.split("/")[-1].split("?")[0]
    elif "youtube.com" in url:
        return url.split("v=")[1].split("&")[0]
    return None

# User-provided fetch_transcript to avoid list_transcripts
from youtube_transcript_api import YouTubeTranscriptApi
def fetch_transcript(video_id):
    try:
        data = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([x["text"] for x in data])
    except Exception as e:
        print(f"Error: {e}")
        return None

def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.create_documents([text])

### Re-checking Transcripts for a New Video

Let's investigate the new video ID `7kFXdQACKlM` to see if transcripts are available, despite the error message you received during processing.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound

video_id_to_debug_new = "7kFXdQACKlM"

try:
    transcript_list_new = YouTubeTranscriptApi.list_transcripts(video_id_to_debug_new)
    print(f"Available transcripts for video ID: {video_id_to_debug_new}")
    for transcript in transcript_list_new:
        print(f"- Language: {transcript.language}, Code: {transcript.language_code}, Is Generated: {transcript.is_generated}")
except NoTranscriptFound:
    print(f"No transcripts found for video ID: {video_id_to_debug_new}")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=7kFXdQACKlM! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load Flan-T5 model and tokenizer directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# The previous 'generator' pipeline is replaced by direct model/tokenizer usage
# We will use this model and tokenizer in the ask_question function

vectordb = None

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
process_button = widgets.Button(description="Process Podcasts", button_style="success")
process_output = widgets.Output()

def process_podcasts(b):
    global vectordb
    all_docs = []

    with process_output:
        clear_output()
        print("Processing podcasts...\n")

        for idx, widget in enumerate(url_inputs):
            url = widget.value.strip()

            if url:
                video_id = extract_video_id(url)
                if not video_id:
                    print(f"Invalid YouTube URL: {url}. Skipping Podcast {idx+1}.")
                    continue

                print(f"Fetching transcript for Podcast {idx+1} (ID: {video_id})...")
                # Call the new fetch_transcript function
                transcript_text = fetch_transcript(video_id)

                if transcript_text:
                    try:
                        docs = chunk_text(transcript_text)

                        for doc in docs:
                            doc.metadata = {"source": f"Podcast {idx+1}", "url": url}

                        all_docs.extend(docs)
                        print(f"Podcast {idx+1} processed successfully.")
                    except Exception as e:
                        print(f"Error chunking/processing transcript for Podcast {idx+1}: {e}")
                else:
                    print(f"Skipping Podcast {idx+1} due to transcript fetching error.")

        if all_docs:
            vectordb = Chroma.from_documents(all_docs, embedding_model)
            print("\nAll podcasts indexed successfully!")
        else:
            print("No valid podcasts were processed.")

process_button.on_click(process_podcasts)
display(process_button, process_output)

Button(button_style='success', description='Process Podcasts', style=ButtonStyle())

Output()

In [ ]:
question_input = widgets.Text(
    placeholder="Ask a question about the podcasts",
    description="Question:",
    layout=widgets.Layout(width='90%')
)

ask_button = widgets.Button(description="Ask", button_style="info")
answer_output = widgets.Output()

def ask_question(b):
    with answer_output:
        clear_output()

        if vectordb is None:
            print("Please process podcasts first.")
            return

        query = question_input.value.strip()

        if not query:
            print("Enter a question.")
            return

        results = vectordb.similarity_search(query, k=3)
        context = "\n\n".join([doc.page_content for doc in results])

        prompt = f"""
        Answer the question based on the context below.

        Context:
        {context}

        Question:
        {query}

        Answer:
        """
        # Encode the prompt
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids

        # Generate output
        outputs = model.generate(input_ids, max_new_tokens=256)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        sources = list(dict.fromkeys([doc.metadata["source"] for doc in results]))

        print("Answer:\n")
        print(response)
        print("\nSources:")
        for source in sources:
            print(f"- {source}")

ask_button.on_click(ask_question)
display(question_input, ask_button, answer_output)

Text(value='', description='Question:', layout=Layout(width='90%'), placeholder='Ask a question about the podc…

Button(button_style='info', description='Ask', style=ButtonStyle())

Output()